In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

In [ ]:
#load csvs
benign_path = r"C:\Users\t0eur\OneDrive\Documents\CiCIOMT Data\flows\Benign_train.pcap_Flow.csv"
connect_path = r"C:\Users\t0eur\OneDrive\Documents\CiCIOMT Data\flows\MQTT-DDoS-Connect_Flood_train.pcap_Flow.csv"
publish_path = r"C:\Users\t0eur\OneDrive\Documents\CiCIOMT Data\flows\MQTT-DDoS-Publish_Flood_train.pcap_Flow.csv"

In [ ]:
#features list
features = [
        'Protocol', 'Flow Duration',
        'Flow Bytes/s', 'Flow Packets/s', 'Down/Up Ratio',
        'Total Fwd Packet', 'Total Bwd packets',
        'Fwd Header Length', 'Bwd Header Length',
        'FWD Init Win Bytes', 'Bwd Init Win Bytes',
        'Fwd Seg Size Min', 'Fwd Segment Size Avg',
        'Subflow Fwd Bytes', 'Subflow Bwd Bytes',
        'Packet Length Min', 'Packet Length Mean',
        'Packet Length Max', 'Packet Length Std',
        'Packet Length Variance', 'Average Packet Size',
        'Fwd IAT Mean', 'Bwd IAT Mean',
        'Fwd IAT Total', 'Bwd IAT Total',
        'Flow IAT Mean',
        'SYN Flag Count', 'ACK Flag Count',
        'PSH Flag Count', 'RST Flag Count', 'FIN Flag Count'
    ]

In [ ]:
#global scaler
df_all = pd.concat([
    pd.read_csv(benign_path)[features],
    pd.read_csv(connect_path)[features],
    pd.read_csv(publish_path)[features]
])

df_all = df_all.replace([np.inf], 1e10)
df_all = df_all.replace([-np.inf], -1e10)
df_all = df_all.fillna(0)

scaler = MinMaxScaler()
scaler.fit(df_all)
print(f"Scaler fitted on {len(df_all):,} total flows")

In [ ]:
#load and preprocess
def load_and_scale(path, class_label, features, scaler):
    """Load CSV, scale features, return (X, y)."""
    df = pd.read_csv(path).replace([np.inf, -np.inf], np.nan).fillna(0) 
    
    # Scale features
    X = scaler.transform(df[features])
    y = np.full(len(X), class_label)
    
    return X, y

print("\nLoading individual datasets...")
X_benign, y_benign = load_and_scale(benign_path, class_label=0, features=features, scaler=scaler)
print(f"Benign: {X_benign.shape}")

X_connect, y_connect = load_and_scale(connect_path, class_label=1, features=features, scaler=scaler)
print(f"Connect: {X_connect.shape}")

X_publish, y_publish = load_and_scale(publish_path, class_label=1, features=features, scaler=scaler)
print(f"Publish: {X_publish.shape}")

In [ ]:
#combine DDoS traffic and undersample to match benign dataset

print("\nCombining DDoS attacks...")
X_attack = np.concatenate([X_connect, X_publish])
y_attack = np.ones(len(X_attack))
print(f"Total DDoS flows: {len(X_attack):,}")

n_samples_per_class = min(len(X_benign), len(X_attack), 100000)

print(f"\nSampling {n_samples_per_class:,} flows per class...")

# Randomly sample from each class
from sklearn.utils import resample

X_benign_sampled = resample(X_benign, n_samples=n_samples_per_class, random_state=42)
y_benign_sampled = np.zeros(n_samples_per_class)

X_attack_sampled = resample(X_attack, n_samples=n_samples_per_class, random_state=42)
y_attack_sampled = np.ones(n_samples_per_class)

print(f"\n✓ Balanced dataset created:")
print(f"  Benign: {len(X_benign_sampled):,}")
print(f"  DDoS:   {len(X_attack_sampled):,}")
print(f"  Total:  {len(X_benign_sampled) + len(X_attack_sampled):,}")

In [ ]:
print("Final dataset...")
X = np.concatenate([X_benign_sampled, X_attack_sampled])
y = np.concatenate([y_benign_sampled, y_attack_sampled])

# Shuffle the data
shuffle_idx = np.random.RandomState(42).permutation(len(X))
X = X[shuffle_idx]
y = y[shuffle_idx]

print(f"Final balanced dataset: {X.shape}")
print(f"  Class 0 (Benign): {np.sum(y==0):,} ({np.mean(y==0)*100:.1f}%)")
print(f"  Class 1 (DDoS):   {np.sum(y==1):,} ({np.mean(y==1)*100:.1f}%)")

In [ ]:
#split train/test
print("Splitting train/test (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

print(f"\nClass weights:")
print(f"  Benign (0): {class_weight_dict[0]:.4f}")
print(f"  DDoS (1):   {class_weight_dict[1]:.4f}")

In [ ]:
noise_level = 0.5
X_train_noisy = X_train + np.random.normal(0, noise_level, X_train.shape)
X_train_noisy = np.clip(X_train_noisy, 0, 1)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1,
    verbose=1
)
rf.fit(X_train_noisy, y_train)

In [ ]:
#evaluate
y_pred_train = rf.predict(X_train)
y_pred_test = rf.predict(X_test)

train_acc = accuracy_score(y_train, y_pred_train)
print(f"\nTrain Accuracy: {train_acc*100:.2f}%")

test_acc = accuracy_score(y_test, y_pred_test)
print(f"Test Accuracy: {test_acc*100:.2f}%")

# Confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_test, target_names=['Benign', 'DDoS'], digits=4))

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign', 'DDoS'],
            yticklabels=['Benign', 'DDoS'],
            cbar=True, 
            square=True)

plt.title('Random Forest - Confusion Matrix', fontsize=16, weight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()